# Notebook 3 — Hypothesis Testing with Permutation Tests

In Notebook 2 we measured conservation across the prestin alignment.
Now we ask a sharper question: **do echolocating species converge on
the same amino acid at specific positions, more than expected by chance?**

To answer this, we need to understand **hypothesis testing** and
**p-values**. We'll build the concepts from scratch with simple
examples, then apply them to our real data.

**What you'll learn today:**
1. What a p-value actually means — through colored balls and shuffling
2. How **permutation tests** work as a general framework
3. How to test for **convergent evolution** at alignment positions
4. How to deal with **multiple testing**

---
## 1. Setup

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import math

sns.set_theme()
np.random.seed(42)

DATA = os.path.join('..', 'data')
SUB_DIR = os.path.join(DATA, 'subfamilies')

def read_fasta(path):
    seqs = {}
    header = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                header = line[1:].split()[0]
                seqs[header] = ''
            elif header:
                seqs[header] += line
    return seqs

---
## 2. Probabilities with colored balls

Imagine a bag containing **7 blue balls** and **3 red balls** (10 total).
You draw 3 balls **without replacement**. All 3 turn out to be red.

**Question:** What is the probability of that happening by chance?

### 2.1 The analytical answer

We can compute this exactly using combinatorics:

$$P(\text{3 red out of 3 draws}) = \frac{\binom{3}{3} \cdot \binom{7}{0}}{\binom{10}{3}}$$

In [ ]:
from math import comb

p_exact = comb(3, 3) * comb(7, 0) / comb(10, 3)
print(f"Exact probability: {p_exact:.4f}")
print(f"That's 1 in {1/p_exact:.0f}")

So there's about a 0.83% chance of drawing all 3 red balls. That's
pretty unlikely — we'd call this result **statistically significant**
at the 5% level (p < 0.05).

### 2.2 The simulation approach

But what if the problem were more complex and we couldn't compute the
answer analytically? We can **simulate** it:
1. Create a bag with 7 blue and 3 red balls
2. Draw 3 balls at random
3. Check if all 3 are red
4. Repeat many times and count how often it happens

In [ ]:
bag = ['blue'] * 7 + ['red'] * 3

n_simulations = 100000
all_red_count = 0

for _ in range(n_simulations):
    draw = np.random.choice(bag, size=3, replace=False)
    if all(ball == 'red' for ball in draw):
        all_red_count += 1

p_simulated = all_red_count / n_simulations
print(f"Simulated probability: {p_simulated:.4f}")
print(f"Exact probability:     {p_exact:.4f}")

The simulation matches the exact answer. This is the core idea
behind **permutation testing**: when the math is hard, simulate.

### ✏️ Exercise 1

1. What if the bag had 5 blue and 5 red, and you drew 3?
   Compute both the exact probability of getting 3 red and the
   simulated probability. Is it still significant at p < 0.05?

2. What about 8 blue and 2 red, drawing 2? (All red.)

3. How many simulations do you need for the result to stabilize?
   Try 100, 1000, 10000, 100000. When does the third decimal place
   stop changing?

In [ ]:
# Your code here

---
## 3. From balls to hypothesis testing

The colored balls example illustrates the logic of a **hypothesis test**:

1. **Null hypothesis (H₀):** The balls were drawn randomly — color
   doesn't matter.
2. **Observation:** All 3 drawn balls are red.
3. **P-value:** The probability of seeing this result (or more extreme)
   **if H₀ is true**.
4. **Decision:** If the p-value is small (< 0.05 by convention),
   we **reject H₀** — the result is unlikely to be due to chance.

A **permutation test** generalizes this: instead of computing the
exact probability, we **shuffle the labels** (which balls are "drawn")
many times and see how often the shuffled result is as extreme as
what we observed.

### 3.1 Permutation test on the balls

In [ ]:
# The bag, as a numpy array for speed
bag = np.array(['blue'] * 7 + ['red'] * 3)

# Our observation: we drew positions 7, 8, 9 (the 3 red balls)
# The "statistic" is: number of red balls in the draw
observed_red = 3

# Permutation test: shuffle which 3 positions we "draw"
n_perm = 100000
null_counts = np.zeros(n_perm)

for i in range(n_perm):
    draw_idx = np.random.choice(10, size=3, replace=False)
    null_counts[i] = sum(bag[draw_idx] == 'red')

p_perm = np.mean(null_counts >= observed_red)
print(f"Permutation p-value: {p_perm:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
values, counts = np.unique(null_counts, return_counts=True)
ax.bar(values, counts / n_perm, color='lightblue', edgecolor='white', width=0.6)
ax.axvline(observed_red, color='red', linewidth=2, linestyle='--',
           label=f'Observed = {observed_red}')
ax.set_xlabel('Number of red balls drawn')
ax.set_ylabel('Probability')
ax.set_title('Null distribution: how many red balls by chance?')
ax.set_xticks([0, 1, 2, 3])
ax.legend()
plt.tight_layout()
plt.show()

The red dashed line shows our observation (3 red). The histogram
shows what happens under random drawing. Since our observation is
in the far tail, the p-value is small → it's unlikely by chance.

---
## 4. Amino acid frequencies in proteins

Let's bring this closer to biology. Amino acids are **not equally
frequent** in proteins. Some (like Leucine, L) are very common;
others (like Tryptophan, W) are rare.

In [ ]:
# Load our prestin alignment
prestin = read_fasta(os.path.join(SUB_DIR, 'SLC26A5.trim.fa'))
headers = list(prestin.keys())
matrix = [list(seq) for seq in prestin.values()]
aln = pd.DataFrame(matrix, index=headers)

# Overall amino acid frequencies in prestin (ignoring gaps)
all_aas = []
for seq in prestin.values():
    all_aas.extend([aa for aa in seq if aa != '-'])

aa_freqs = Counter(all_aas)
total_aa = sum(aa_freqs.values())

print(f"Total amino acids: {total_aa}")
print(f"\n{'AA':>3s}  {'Count':>7s}  {'Freq':>6s}")
print('-' * 20)
for aa, count in aa_freqs.most_common():
    print(f"  {aa:>2s}  {count:>7d}  {count/total_aa:>6.3f}")

### 4.1 How surprising is a conserved Tryptophan?

Suppose we find a position where **all species** have Tryptophan (W).
Tryptophan is rare (~1-2% of amino acids in most proteins).
Is it surprising that all ~30 species agree on W at one position?

If amino acids were placed **randomly** according to their overall
frequencies, the probability of all N species having W would be
$p_W^N$.

In [ ]:
# Frequency of W in prestin
p_W = aa_freqs['W'] / total_aa
n_species = aln.shape[0]

p_all_W = p_W ** n_species
print(f"Frequency of W in prestin: {p_W:.4f}")
print(f"P(all {n_species} species have W): {p_all_W:.2e}")

That's an astronomically small number — a conserved Tryptophan is
almost certainly there for a reason (structural or functional
constraint), not by chance.

### ✏️ Exercise 2

1. What about Leucine (L), the most common amino acid? What's the
   probability of all species having L at one position by chance?

2. Is that probability still very small? What does this tell you about
   the difference between **conservation by constraint** and
   **conservation by frequency**?

3. Find a position in the alignment that is perfectly conserved
   (entropy = 0). What amino acid is it? How surprising is its
   conservation given its overall frequency?

In [ ]:
# Your code here

---
## 5. Testing for convergent evolution

Now the real question. **Conservation** means all species agree.
**Convergence** means that specifically the **echolocating** species
agree — even though they are not closely related.

At a truly convergent position:
- Echolocators (bats + dolphin) share one amino acid
- Non-echolocators have something different

We need a test statistic and a way to compute its p-value.

### 5.1 The test statistic: agreement score

For a group of species at one alignment position:

$$\text{agreement} = \frac{\text{count of the most common amino acid in the group}}{\text{group size}}$$

- If all echolocators have the same amino acid: agreement = 1.0
- If they're split: agreement < 1.0

In [ ]:
# Load species classification
species_df = pd.read_csv(
    os.path.join(DATA, 'species_classification.tsv'),
    sep='\t', comment='#',
    names=['taxid', 'species', 'group', 'echolocates', 'notes']
)
species_df['taxid'] = species_df['taxid'].astype(str)
echo_lookup = dict(zip(species_df['taxid'], species_df['echolocates']))

# Build boolean index
taxids = [h.split('.')[0] for h in aln.index]
is_echo = pd.Series(
    [echo_lookup.get(t) == 'yes' for t in taxids],
    index=aln.index
)
echo_aln = aln[is_echo]
other_aln = aln[~is_echo]
n_echo = int(is_echo.sum())

print(f"Echolocators: {n_echo}")
print(f"Non-echolocators: {(~is_echo).sum()}")

In [ ]:
def agreement_score(amino_acids):
    """Fraction of a group sharing the most common amino acid."""
    residues = [aa for aa in amino_acids if aa != '-']
    if len(residues) == 0:
        return 0.0
    most_common_count = Counter(residues).most_common(1)[0][1]
    return most_common_count / len(residues)

In [ ]:
# Example at one position
pos = 300
print(f"Position {pos}:")
print(f"  Echolocators:   {list(echo_aln[pos].values)}")
print(f"  Agreement:      {agreement_score(list(echo_aln[pos])):.3f}")
print(f"  Non-echolocators: {list(other_aln[pos].values)}")
print(f"  Agreement:      {agreement_score(list(other_aln[pos])):.3f}")

### 5.2 Permutation test at one position

Same logic as the colored balls:
1. Compute the **observed** agreement among echolocators
2. **Shuffle**: randomly label N species as "echolocators"
3. Recompute agreement for the fake echolocators
4. Repeat → build null distribution → p-value

### ✏️ Exercise 3

**Write the permutation test yourself.** Fill in the code below:

```python
pos = 300
all_aas = list(aln[pos])
n_total = len(all_aas)

# Step 1: observed score
echo_aas = [all_aas[i] for i in range(n_total) if is_echo.iloc[i]]
obs_score = agreement_score(echo_aas)

# Step 2-4: permutation loop
n_perm = 10000
null_scores = np.zeros(n_perm)
for i in range(n_perm):
    # YOUR CODE: randomly pick n_echo indices
    # compute agreement for the random group
    pass

# Step 5: p-value
p_value = ???
```

In [ ]:
# YOUR CODE HERE
pos = 300

### 5.3 Solution

In [ ]:
pos = 300
all_aas = list(aln[pos])
n_total = len(all_aas)

# Observed
echo_aas = [all_aas[i] for i in range(n_total) if is_echo.iloc[i]]
obs_score = agreement_score(echo_aas)
print(f"Observed agreement at position {pos}: {obs_score:.3f}")

# Permutations
n_perm = 10000
null_scores = np.zeros(n_perm)

for i in range(n_perm):
    fake_idx = np.random.choice(n_total, size=n_echo, replace=False)
    fake_aas = [all_aas[j] for j in fake_idx]
    null_scores[i] = agreement_score(fake_aas)

p_value = np.mean(null_scores >= obs_score)
print(f"P-value: {p_value:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_scores, bins=30, color='lightblue', edgecolor='white')
ax.axvline(obs_score, color='red', linewidth=2,
           label=f'Observed = {obs_score:.3f}')
ax.set_xlabel('Agreement score')
ax.set_ylabel('Count')
ax.set_title(f'Permutation test at position {pos} (p = {p_value:.4f})')
ax.legend()
plt.tight_layout()
plt.show()

### ✏️ Exercise 4

1. Try different positions. Find one with a **very low p-value**
   (p < 0.01) and one with a **high p-value** (p > 0.5).

2. At a perfectly conserved position (entropy = 0), what happens?
   Is the p-value low? Why or why not?
   *(Think about it: if ALL species have the same amino acid, is
   echolocator agreement special?)*

3. What if you increase the number of permutations to 50,000?
   Does the p-value change much?

In [ ]:
# Your code here

---
## 6. Scanning the whole alignment

Now let's run the test at every position and find where convergence
signals appear.

In [ ]:
def convergence_pvalue(aln, is_echo, pos, n_perm=1000):
    """Permutation test for convergence at one alignment position."""
    all_aas = list(aln[pos])
    n_echo = int(is_echo.sum())
    n_total = len(all_aas)

    # Observed
    echo_aas = [all_aas[i] for i in range(n_total) if is_echo.iloc[i]]
    obs = agreement_score(echo_aas)

    # Null distribution
    count_extreme = 0
    for _ in range(n_perm):
        idx = np.random.choice(n_total, size=n_echo, replace=False)
        null_score = agreement_score([all_aas[j] for j in idx])
        if null_score >= obs:
            count_extreme += 1

    return obs, count_extreme / n_perm

In [ ]:
n_pos = aln.shape[1]
results = []

print(f"Scanning {n_pos} positions (1000 permutations each)...")
for pos in range(n_pos):
    score, pval = convergence_pvalue(aln, is_echo, pos, n_perm=1000)
    results.append({'position': pos, 'agreement': score, 'pvalue': pval})
    if (pos + 1) % 100 == 0:
        print(f"  ... {pos + 1}/{n_pos}")

results_df = pd.DataFrame(results)
print("Done!")

In [ ]:
n_sig_05 = (results_df['pvalue'] < 0.05).sum()
n_sig_01 = (results_df['pvalue'] < 0.01).sum()
print(f"Positions with p < 0.05:  {n_sig_05} / {n_pos}")
print(f"Positions with p < 0.01:  {n_sig_01} / {n_pos}")

### 6.1 Manhattan plot

In [ ]:
# Load entropies from NB02 (recompute here for self-containment)
def shannon_entropy(column):
    residues = [aa for aa in column if aa != '-']
    if not residues:
        return 0.0
    counts = Counter(residues)
    total = len(residues)
    return -sum((c/total) * math.log2(c/total) for c in counts.values())

entropies = [shannon_entropy(list(aln[pos])) for pos in range(n_pos)]

In [ ]:
# Replace p=0 to avoid log(0)
pvals = results_df['pvalue'].replace(0, 1 / 10001)
neg_log_p = -np.log10(pvals)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

# Top: conservation
axes[0].fill_between(range(n_pos), entropies, alpha=0.4, color='steelblue')
axes[0].plot(range(n_pos), entropies, linewidth=0.5, color='steelblue')
axes[0].set_ylabel('Shannon entropy\n(bits)')
axes[0].set_title('Prestin (SLC26A5) — conservation vs. convergence signal')

# Bottom: convergence p-values
axes[1].scatter(range(n_pos), neg_log_p, s=5, color='coral', alpha=0.7)
axes[1].axhline(-np.log10(0.05), color='gray', linewidth=1, linestyle='--',
                label='p = 0.05')
axes[1].axhline(-np.log10(0.01), color='gray', linewidth=1, linestyle=':',
                label='p = 0.01')
axes[1].set_xlabel('Alignment position')
axes[1].set_ylabel('$-\\log_{10}$(p-value)')
axes[1].legend()
axes[1].set_xlim(0, n_pos)

plt.tight_layout()
plt.show()

### 6.2 Top convergent positions

In [ ]:
top = results_df.sort_values('pvalue').head(20)

print(f"{'Position':>10s}  {'Agreement':>10s}  {'P-value':>10s}  {'Echo AAs'}")
print('-' * 60)
for _, row in top.iterrows():
    pos = int(row['position'])
    echo_aas = ''.join(echo_aln[pos].values)
    other_aas_top3 = other_aln[pos].value_counts().head(3).to_dict()
    print(f"  {pos:>8d}  {row['agreement']:>10.3f}  {row['pvalue']:>10.4f}  "
          f"{echo_aas}  (others: {other_aas_top3})")

### ✏️ Exercise 5

1. **False positives:** We tested ~700 positions at p < 0.05. By
   chance alone, how many would we *expect* to be "significant"?
   ```python
   expected_false_positives = n_pos * 0.05
   ```

2. **Bonferroni correction:** Divide the threshold by the number
   of tests. How many positions survive?
   ```python
   bonf = 0.05 / n_pos
   n_bonf = (results_df['pvalue'] < bonf).sum()
   ```

3. Look at the top hits. At those positions, do echolocating **bats**
   and echolocating **cetaceans** have the **same** amino acid?
   That would be true convergence — two independent lineages arriving
   at the same solution.

4. Are convergent positions in conserved or variable regions? Compare
   with the entropy plot.

In [ ]:
# Your code here

---
## 7. Summary and connection to the literature

### What we did
1. Learned what p-values mean through the **colored balls** example
2. Understood **permutation tests** as a general, simulation-based
   approach to hypothesis testing
3. Applied a convergence test to every position in the prestin alignment
4. Found positions where echolocators agree more than expected by chance

### What a rigorous analysis would add
- **Phylogenetic correction:** Closely related species share amino acids
  by inheritance. A proper test accounts for the species tree.
- **Ancestral state reconstruction:** Distinguish true convergence
  (independent gains) from shared ancestral states.
- **Better statistics:** FDR correction (Benjamini-Hochberg) instead
  of Bonferroni.
- **Domain mapping:** Are convergent sites in functionally important
  regions? (This is your assignment!)

### Connection to Parker et al. (2013)
[Parker et al.](https://www.nature.com/articles/nature12511) did a
genome-wide analysis across 22 mammals and found widespread convergent
evolution in echolocating species — not just in prestin but across
many genes involved in hearing. Our simplified analysis recapitulates
their core finding for prestin.

---
## 8. Save results for the assignment

Save the convergence results so the assignment notebook can use them.

In [ ]:
results_path = os.path.join(DATA, 'prestin_convergence_results.csv')
results_df.to_csv(results_path, index=False)
print(f"Saved convergence results → {results_path}")

# Save entropy values too
entropy_df = pd.DataFrame({
    'position': range(n_pos),
    'entropy': entropies
})
entropy_path = os.path.join(DATA, 'prestin_entropy.csv')
entropy_df.to_csv(entropy_path, index=False)
print(f"Saved entropy values → {entropy_path}")